<a href="https://colab.research.google.com/github/syahidmid/google-colab/blob/main/wordpress/category/create_categories.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📂 Bulk Create WordPress Categories
Script ini membuat kategori parent & child secara otomatis ke WordPress via REST API.

**Website:** balitouristic.com  
**Struktur:** 6 parent categories, beberapa dengan sub-categories

In [ ]:
import requests
import time
from requests.auth import HTTPBasicAuth

# Jika menggunakan Google Colab
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    userdata = None

# ── Endpoint & Kredensial ──────────────────────────────────────────────────────
WORDPRESS_URL = "https://www.balitouristic.com/wp-json/wp/v2/categories"

if userdata:
    username = userdata.get('admin_bali')
    password = userdata.get('pass_bali')
else:
    username = None
    password = None

if not username or not password:
    raise ValueError("❌ Kredensial tidak ditemukan di userdata Colab.")

auth = HTTPBasicAuth(username, password)
print("✅ Kredensial berhasil dimuat.")

In [ ]:
# ── Struktur Kategori ──────────────────────────────────────────────────────────
# Format: list of dict dengan key 'name' dan optional 'children'

CATEGORY_STRUCTURE = [
    {
        "name": "Places in Bali",
        "children": [
            "Badung",
            "Bangli",
            "Buleleng",
            "Denpasar",
            "Gianyar",
            "Jembrana",
            "Karangasem",
            "Klungkung",
            "Nusa Penida",
            "Tabanan"
        ]
    },
    {
        "name": "Bali Adventure & Activities",
        "children": [
            "Water Activities",
            "Land Activities",
            "Cultural Activities",
            "Romantic & Honeymoon"
        ]
    },
    {
        "name": "Bali Beaches",
        "children": []
    },
    {
        "name": "Bali Airport Transfer & Car Rental",
        "children": []
    },
    {
        "name": "Bali Travel Guide & Tips",
        "children": [
            "Bali Itinerary & Packages",
            "Bali Culture & Temples"
        ]
    },
    {
        "name": "Bali SPA",
        "children": []
    }
]

print(f"📋 Total parent categories  : {len(CATEGORY_STRUCTURE)}")
total_children = sum(len(c['children']) for c in CATEGORY_STRUCTURE)
print(f"📋 Total child categories   : {total_children}")
print(f"📋 Total semua categories   : {len(CATEGORY_STRUCTURE) + total_children}")

In [ ]:
# ── Fungsi Helper ──────────────────────────────────────────────────────────────

def create_category(name, parent_id=0):
    """Buat satu category. Return ID jika sukses, None jika gagal."""
    payload = {"name": name, "parent": parent_id}
    response = requests.post(WORDPRESS_URL, auth=auth, json=payload)

    if response.status_code == 201:
        data = response.json()
        return data["id"]
    elif response.status_code == 400 and "term_exists" in response.text:
        # Category sudah ada — ambil ID yang existing
        existing_id = response.json().get("data", {}).get("term_id")
        return existing_id
    else:
        print(f"      ❌ Gagal: {response.status_code} — {response.text[:120]}")
        return None


def slug_from_name(name):
    """Buat slug sederhana dari nama."""
    return name.lower().replace(" ", "-").replace("&", "and")


print("✅ Fungsi helper siap.")

In [ ]:
# ── Eksekusi Pembuatan Kategori ────────────────────────────────────────────────

results = []   # Menyimpan log hasil
success = 0
skipped = 0
failed  = 0

print("🚀 Mulai membuat categories...\n")
print("=" * 55)

for parent in CATEGORY_STRUCTURE:
    parent_name = parent["name"]
    print(f"\n📁 Parent: {parent_name}")

    parent_id = create_category(parent_name, parent_id=0)

    if parent_id:
        print(f"   ✅ Berhasil dibuat  (ID: {parent_id})")
        success += 1
        results.append({"name": parent_name, "type": "parent", "id": parent_id, "status": "ok"})
    else:
        print(f"   ❌ Gagal membuat parent category.")
        failed += 1
        results.append({"name": parent_name, "type": "parent", "id": None, "status": "failed"})
        continue   # Skip children kalau parent gagal

    # Buat child categories
    for child_name in parent["children"]:
        time.sleep(0.3)  # Hindari rate limiting
        child_id = create_category(child_name, parent_id=parent_id)

        if child_id:
            print(f"      └─ ✅ Child: {child_name}  (ID: {child_id})")
            success += 1
            results.append({"name": child_name, "type": "child", "id": child_id, "parent_id": parent_id, "status": "ok"})
        else:
            print(f"      └─ ❌ Gagal: {child_name}")
            failed += 1
            results.append({"name": child_name, "type": "child", "id": None, "parent_id": parent_id, "status": "failed"})

    time.sleep(0.3)

print("\n" + "=" * 55)
print(f"\n🎯 Selesai!")
print(f"   ✅ Berhasil : {success}")
print(f"   ❌ Gagal    : {failed}")

In [ ]:
# ── Tampilkan Ringkasan Hasil ──────────────────────────────────────────────────

print("\n📊 RINGKASAN HASIL\n")
print(f"{'Nama':<40} {'Tipe':<8} {'ID':<8} {'Status'}")
print("-" * 68)

for r in results:
    indent = "  └─ " if r["type"] == "child" else ""
    status_icon = "✅" if r["status"] == "ok" else "❌"
    print(f"{indent + r['name']:<40} {r['type']:<8} {str(r.get('id','–')):<8} {status_icon}")